# Challenge ANNITIA — Modèle Prédictif de Survie MASLD

**Auteur :** Samuel Yevi  
**Challenge :** ANNITIA par TRUSTII / ICAN  
**Objectif :** Prédire deux endpoints de survie à partir de NITs longitudinaux :
- `risk_hepatic_event` — risque d'événement hépatique majeur (MASH, décompensation, CHC)
- `risk_death` — risque de décès toutes causes

**Score :** `0.7 × C-index hépatique + 0.3 × C-index décès`

---

## Architecture du pipeline

```
CSV wide (1253 patients × 22 visites × 18 features)
        |
        ▼
Feature Engineering temporel (162 features)
  • Statistiques par feature : last, first, max, min, mean, std, slope, delta
  • Accélération temporelle  : slope_recent, accel (2e moitié des visites)
  • Ratios cliniques MASLD   : FIB-4, AST/ALT, GGT/ALT, FibroScan×FibroTest
  • Patterns de données manquantes : obs_rate, n_features_last_visit
        |
        ▼
XGBoost Cox PH  (objectif survival:cox natif)
  • 5-fold stratified CV + early stopping
  • 5 seeds → moyenne des prédictions (bagging)
        |
        ▼
Prédictions de risque [0, +∞)
```

**Modèle complémentaire :** K-Mamba SSM — réseau de neurones séquentiel (State Space Model)
entraîné sur les trajectoires longitudinales brutes. Implémenté en C pur avec backpropagation end-to-end.

## 1. Imports & Configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.model_selection import StratifiedKFold
from pathlib import Path

# ─── Chemins des données Trustii ─────────────────────────────────────────────
# Adapter ces chemins à l'environnement Trustii si nécessaire.
# Candidats courants : 'data/train.csv', '../data/train.csv', 'train.csv'
_candidates_train = [
    'data/processed_wide/train.csv',
    'data/train.csv',
    '../data/train.csv',
    'train.csv',
]
_candidates_test = [
    'data/processed_wide/val_test.csv',
    'data/val_test.csv',
    'data/test.csv',
    '../data/val_test.csv',
    '../data/test.csv',
    'val_test.csv',
    'test.csv',
]

def _find_csv(candidates):
    for p in candidates:
        if Path(p).exists():
            return p
    raise FileNotFoundError(
        f"Fichier de données introuvable. Candidats essayés : {candidates}\n"
        "Mettez à jour TRAIN_CSV / TEST_CSV manuellement.")

TRAIN_CSV  = _find_csv(_candidates_train)
TEST_CSV   = _find_csv(_candidates_test)
OUTPUT_CSV = 'submission_1.csv'
N_VISITS   = 22
N_FOLDS    = 5
SEEDS      = [42, 123, 777, 2024, 31415]

print(f'Train CSV : {TRAIN_CSV}')
print(f'Test  CSV : {TEST_CSV}')
print(f'XGBoost   : {xgb.__version__}')

## 2. Chargement des données

In [ ]:
df_train = pd.read_csv(TRAIN_CSV)
df_test  = pd.read_csv(TEST_CSV)

print(f'Train : {df_train.shape[0]} patients, {df_train.shape[1]} colonnes')
print(f'Test  : {df_test.shape[0]} patients')

age_cols   = [f'Age_v{v}' for v in range(1, N_VISITS+1)]

event_hep  = df_train['evenements_hepatiques_majeurs'].fillna(0).astype(int).values
time_hep   = df_train['evenements_hepatiques_age_occur'].where(
                 df_train['evenements_hepatiques_majeurs'] == 1,
                 df_train[age_cols].max(axis=1)
             ).fillna(1e-3).values

event_dth  = df_train['death'].fillna(0).astype(int).values
time_dth   = df_train['death_age_occur'].where(
                 df_train['death'] == 1,
                 df_train[age_cols].max(axis=1)
             ).fillna(1e-3).values

print(f'\nÉvénements hépatiques : {event_hep.sum()} ({100*event_hep.mean():.1f}%)')
print(f'Décès                 : {event_dth.sum()} ({100*event_dth.mean():.1f}%)')
print(f'Suivi médian          : {np.median(df_train[age_cols].max(axis=1) - df_train["Age_v1"]):.1f} ans')

## 3. Feature Engineering temporel

Chaque patient dispose de mesures répétées sur jusqu'à 22 visites. Pour chaque feature dynamique
(BMI, ALT, AST, bilirubine, cholestérol, GGT, glucose, plaquettes, triglycérides, Aixplorer, FibroTest, FibroScan),
on extrait :

| Statistique | Description |
|-------------|-------------|
| `last`, `first` | Valeur à la dernière / première visite |
| `max`, `min`, `mean`, `std` | Statistiques sur toutes les visites |
| `slope` | Tendance linéaire (OLS) sur l'ensemble du suivi |
| `delta` | Variation absolue last − first |
| `slope_recent` | Tendance sur la 2e moitié des visites |
| `accel` | Accélération : mean(2e moitié) − mean(1ère moitié) |
| `obs_rate` | Proportion de visites avec mesure |

**Ratios cliniques MASLD :**
- **FIB-4** = (âge × AST) / (plaquettes × √ALT) — marqueur de fibrose validé
- **De Ritis** (AST/ALT) > 1 → fibrose avancée probable
- **GGT/ALT** — lésion hépatocellulaire vs cholestase
- **FibroScan × FibroTest** — concordance des deux mesures de fibrose non-invasives

In [ ]:
# Feature engineering (implémenté dans annitia/lgbm.py)
# Reproduit ici de façon autonome pour la soumission

DYN_PREFIXES = [
    'BMI_v', 'alt_v', 'ast_v', 'bilirubin_v', 'chol_v', 'ggt_v',
    'gluc_fast_v', 'plt_v', 'triglyc_v',
    'aixp_aix_result_BM_3_v', 'fibrotest_BM_2_v', 'fibs_stiffness_med_BM_1_v',
]
DYN_NAMES = [
    'BMI', 'ALT', 'AST', 'bilirubin', 'chol', 'GGT',
    'gluc_fast', 'plt', 'triglyc', 'Aixplorer', 'FibroTest', 'FibroScan',
]
STAT_COLS = ['gender', 'T2DM', 'Hypertension', 'Dyslipidaemia',
             'bariatric_surgery', 'bariatric_surgery_age']

def _slope(values):
    """Pente OLS sur les valeurs non-NaN."""
    mask = ~np.isnan(values)
    if mask.sum() < 2:
        return np.nan
    x = np.where(mask)[0].astype(float)
    y = values[mask]
    x -= x.mean()
    d = (x ** 2).sum()
    return float((x * y).sum() / d) if d > 0 else 0.0

def engineer_features(df):
    rows = []
    for _, row in df.iterrows():
        f = {}
        ages = np.array([
            float(row[f'Age_v{v}']) if f'Age_v{v}' in row.index and not pd.isna(row[f'Age_v{v}'])
            else np.nan for v in range(1, N_VISITS + 1)
        ])
        age_v1   = float(ages[0]) if not np.isnan(ages[0]) else 0.0
        vm       = ~np.isnan(ages)
        n_visits = int(vm.sum())
        last_age = float(ages[vm][-1]) if n_visits > 0 else age_v1
        f['n_visits']  = n_visits
        f['age_v1']    = age_v1
        f['follow_up'] = last_age - age_v1

        for prefix, name in zip(DYN_PREFIXES, DYN_NAMES):
            vals = np.array([
                float(row[f'{prefix}{v}'])
                if f'{prefix}{v}' in row.index and not pd.isna(row[f'{prefix}{v}'])
                else np.nan for v in range(1, N_VISITS + 1)
            ])
            valid = vals[~np.isnan(vals)]
            f[f'{name}_last']     = float(valid[-1])    if len(valid) > 0 else np.nan
            f[f'{name}_first']    = float(valid[0])     if len(valid) > 0 else np.nan
            f[f'{name}_max']      = float(valid.max())  if len(valid) > 0 else np.nan
            f[f'{name}_min']      = float(valid.min())  if len(valid) > 0 else np.nan
            f[f'{name}_mean']     = float(valid.mean()) if len(valid) > 0 else np.nan
            f[f'{name}_std']      = float(valid.std())  if len(valid) > 1 else 0.0
            f[f'{name}_slope']    = _slope(vals)
            f[f'{name}_n_obs']    = int(len(valid))
            f[f'{name}_delta']    = float(valid[-1] - valid[0]) if len(valid) >= 2 else np.nan
            f[f'{name}_obs_rate'] = len(valid) / N_VISITS

            vi = np.where(~np.isnan(vals))[0]
            if len(vi) >= 4:
                mid = len(vi) // 2
                first_half = vals[vi[:mid]]
                second_half = vals[vi[mid:]]
                f[f'{name}_slope_recent'] = _slope(vals[vi[mid:]])
                f[f'{name}_accel'] = float(second_half.mean() - first_half.mean())
            else:
                f[f'{name}_slope_recent'] = np.nan
                f[f'{name}_accel']        = np.nan

        for col in STAT_COLS:
            f[col] = float(row[col]) if col in row.index and not pd.isna(row[col]) else np.nan

        alt   = f.get('ALT_last',      np.nan)
        ast   = f.get('AST_last',      np.nan)
        ggt   = f.get('GGT_last',      np.nan)
        plt_  = f.get('plt_last',      np.nan)
        bili  = f.get('bilirubin_last', np.nan)
        bmi   = f.get('BMI_last',      np.nan)
        fibro = f.get('FibroScan_last', np.nan)
        ftest = f.get('FibroTest_last', np.nan)

        f['AST_ALT_ratio']             = ast / alt if (not any(np.isnan([ast, alt])) and alt != 0) else np.nan
        f['GGT_ALT_ratio']             = ggt / alt if (not any(np.isnan([ggt, alt])) and alt != 0) else np.nan
        f['FIB4']                      = (age_v1 * ast) / (plt_ * np.sqrt(alt)) \
                                         if (not any(np.isnan([age_v1, ast, plt_, alt])) and plt_ > 0 and alt > 0) else np.nan
        f['FibroScan_x_FibroTest']     = fibro * ftest if not any(np.isnan([fibro, ftest])) else np.nan
        f['FibroScan_FibroTest_ratio'] = fibro / ftest if (not any(np.isnan([fibro, ftest])) and ftest != 0) else np.nan
        f['bili_x_AST']                = bili  * ast   if not any(np.isnan([bili, ast]))   else np.nan
        f['fibro_x_age']               = fibro * age_v1 if not np.isnan(fibro)             else np.nan
        f['BMI_x_fibro']               = bmi   * fibro if not any(np.isnan([bmi, fibro]))  else np.nan
        f['n_features_last_visit']     = sum(
            1 for px in DYN_PREFIXES
            if f'{px}{n_visits}' in df.columns and not pd.isna(row.get(f'{px}{n_visits}', np.nan))
        ) if n_visits > 0 else 0

        rows.append(f)
    return pd.DataFrame(rows)

print('Feature engineering en cours...')
X_train = engineer_features(df_train)
X_test  = engineer_features(df_test)
print(f'Features extraites : {X_train.shape[1]}')
print(f'Premières features : {list(X_train.columns[:5])}')
print(f'Dernières features : {list(X_train.columns[-5:])}')

## 4. Modèle XGBoost Cox PH

### Pourquoi XGBoost Cox ?

| Critère | RSF | Cox ElasticNet | **XGBoost Cox** |
|---------|-----|----------------|------------------|
| Gestion censure | ✅ | ✅ | ✅ natif |
| Non-linéarités | ✅ | ❌ | ✅ |
| Interactions features | ✅ | ❌ | ✅ |
| Régularisation | ❌ faible | ✅ L1+L2 | ✅ L1+L2 |
| Données manquantes | ❌ | ❌ | ✅ natif |
| EPV faible (1.3) | ❌ overfit | ✅ | ✅ |

**Objectif `survival:cox`** de XGBoost : implémentation du modèle de Cox Proportionnel par Hazard via gradient boosting.  
Labels : `y = +time` pour les événements, `y = -time` pour les censurés.

### Validation croisée 5-fold stratifiée

Stratification simultanée sur l'événement hépatique ET le décès pour garantir une distribution équilibrée dans chaque fold malgré la rareté des événements (3.8%).

In [ ]:
def c_index_numpy(risks, times, events):
    """C-index de Harrell (implémentation numpy, O(n²))."""
    risks, times, events = np.asarray(risks), np.asarray(times), np.asarray(events)
    concordant = discordant = tied = 0
    for i in range(len(times)):
        if events[i] == 0:
            continue
        for j in range(len(times)):
            if i == j or times[j] <= times[i]:
                continue
            if risks[i] > risks[j]:
                concordant += 1
            elif risks[i] < risks[j]:
                discordant += 1
            else:
                tied += 1
    total = concordant + discordant + tied
    return (concordant + 0.5 * tied) / total if total > 0 else 0.5


XGB_PARAMS = dict(
    objective        = 'survival:cox',
    eval_metric      = 'cox-nloglik',
    tree_method      = 'hist',
    learning_rate    = 0.05,
    max_depth        = 4,
    min_child_weight = 5,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    reg_alpha        = 0.1,
    reg_lambda       = 1.0,
    nthread          = -1,
    verbosity        = 0,
)

feat_names = X_train.columns.tolist()


def train_xgb_cv(X, events, times, seeds=SEEDS, n_folds=N_FOLDS):
    """K-fold stratifié multi-seed → OOF + prédictions test."""
    oof_all, test_all = [], []
    for seed in seeds:
        skf_s = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
        oof_seed  = np.zeros(len(X))
        test_seed = np.zeros(len(X_test))
        y_cox = np.where(events == 1, times, -times).astype(np.float32)
        for tr_idx, va_idx in skf_s.split(X, events):
            X_tr, X_va = X.iloc[tr_idx].values, X.iloc[va_idx].values
            y_tr, y_va = y_cox[tr_idx], y_cox[va_idx]
            dtrain = xgb.DMatrix(X_tr, label=y_tr, feature_names=feat_names)
            dval   = xgb.DMatrix(X_va, label=y_va, feature_names=feat_names)
            dtest  = xgb.DMatrix(X_test.values, feature_names=feat_names)
            model  = xgb.train(
                dict(**XGB_PARAMS, seed=seed),
                dtrain,
                num_boost_round=500,
                evals=[(dval, 'val')],
                early_stopping_rounds=50,
                verbose_eval=False,
            )
            oof_seed[va_idx]  = model.predict(dval)
            test_seed        += model.predict(dtest) / n_folds
        oof_all.append(oof_seed)
        test_all.append(test_seed)
    return np.mean(oof_all, axis=0), np.mean(test_all, axis=0)


print('Entraînement XGBoost Cox — hépatique...')
oof_hep, test_hep = train_xgb_cv(X_train, event_hep, time_hep)
ci_hep = c_index_numpy(oof_hep, time_hep, event_hep)
print(f'  C-index OOF hépatique : {ci_hep:.4f}')

print('Entraînement XGBoost Cox — décès...')
oof_dth, test_dth = train_xgb_cv(X_train, event_dth, time_dth)
ci_dth = c_index_numpy(oof_dth, time_dth, event_dth)
print(f'  C-index OOF décès     : {ci_dth:.4f}')

score_oof = 0.7 * ci_hep + 0.3 * ci_dth
print(f'\nScore OOF (0.7×hep + 0.3×dth) = {score_oof:.4f}')

## 5. Résultats — Comparaison des modèles

| Modèle | C-index Hépatique OOF | C-index Décès OOF | Score OOF | Notes |
|--------|----------------------|-------------------|-----------|-------|
| Cox ElasticNet (baseline) | ~0.77 | ~0.72 | ~0.75 | Features flat, 5 coefs actifs |
| RSF (baseline) | ~0.91 | ~0.93 | ~0.91 | **Train score** — overfit évident |
| LightGBM binary + poids temporels | ~0.78 | ~0.64 | ~0.73 | Objectif non adapté |
| K-Mamba SSM (19K params, 50 epochs) | 0.765 | 0.910 | 0.808 | Trajectoires temporelles |
| **XGBoost Cox multi-seed** | **0.885** | **0.876** | **0.882** | **→ Soumission finale** |

## 6. Interprétabilité — Features les plus importantes

L'importance est mesurée par le **gain total** apporté par chaque feature aux splits des arbres.

In [ ]:
# Entraîner un modèle sur tout le train pour l'importance
y_cox_hep = np.where(event_hep==1, time_hep, -time_hep).astype(np.float32)
dtrain_full = xgb.DMatrix(X_train.values.astype(np.float32), label=y_cox_hep, feature_names=feat_names)
model_full  = xgb.train(dict(**XGB_PARAMS, seed=42), dtrain_full,
                        num_boost_round=300, verbose_eval=False)

imp = pd.Series(model_full.get_score(importance_type='gain')).sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#d62728' if 'Fibro' in f or 'FIB4' in f or 'AST' in f
          else '#1f77b4' for f in imp.index]
ax.barh(imp.index[::-1], imp.values[::-1], color=colors[::-1])
ax.set_xlabel('Gain (importance XGBoost)')
ax.set_title('Top 20 features — Événement hépatique (XGBoost Cox)', fontsize=13)
red_patch  = mpatches.Patch(color='#d62728', label='Marqueurs fibrose / lésion hépatique')
blue_patch = mpatches.Patch(color='#1f77b4', label='Autres features temporelles')
ax.legend(handles=[red_patch, blue_patch])
plt.tight_layout()
plt.savefig('feature_importance_hepatic.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nInterprétation clinique des top features :')
interpretations = {
    'FibroScan_max'        : 'Valeur maximale de rigidité hépatique → fibrose avancée',
    'AST_max'              : 'Pic AST → cytolyse, reflète l\'activité nécro-inflammatoire',
    'FibroScan_mean'       : 'Rigidité hépatique moyenne → état fibrotique global',
    'bili_x_AST'           : 'Bilirubine × AST → marqueur combiné insuffisance hépatique',
    'FibroScan_accel'      : 'Accélération FibroScan (2e moitié suivi) → progression fibrose',
    'FIB4'                 : 'FIB-4 index = (âge×AST)/(plaquettes×√ALT) → standard clinique fibrose',
    'GGT_std'              : 'Variabilité GGT → instabilité cholestase/hépatocellulaire',
}
for feat, interp in list(interpretations.items())[:7]:
    print(f'  {feat:30s} → {interp}')

## 7. Architecture K-Mamba SSM (modèle séquentiel complémentaire)

En parallèle du pipeline XGBoost, nous avons développé un **State Space Model (SSM)** 
inspiré de l'architecture Mamba, implémenté en **C pur** avec backpropagation end-to-end.

### Pourquoi un SSM pour les données longitudinales ?

Les NITs MASLD sont des **séquences temporelles irrégulières** : chaque patient a
entre 1 et 22 visites, avec des intervalles variables. Un SSM traite naturellement :
- L'**ordre temporel** des mesures
- Les **données manquantes** via un masque de visite
- Les **dépendances à long terme** entre les visites distantes

```
Patient [T visites, 18 features]
    ↓  W_feat [18 → 64]  +  W_time [1 → 64]
hidden [T, 64]
    ↓  MambaBlock #1  (MIMO rank=4, state=16)
    ↓  MambaBlock #2
pooling sur le dernier timestep valide
    ↓  W_hepatic [64 → 1]    W_death [64 → 1]
risk_hepatic,  risk_death
```

**Loss de survie :**  
`L = α × ranking_loss + (1-α) × cox_loss`  
avec `α_hépatique = 0.2` (plus de ranking car événements très rares à 3.8%)
et `α_décès = 0.5`.

**Résultats SSM (5-fold OOF, 50 epochs, mimo_rank=4) :**  
C-index hépatique = 0.765, C-index décès = 0.910, **score = 0.808**

Le SSM capture des patterns temporels complémentaires (accélération de la progression
fibrotique visible dans les séquences), mais nécessite plus d'epochs pour atteindre
son plein potentiel avec seulement 47 événements.

## 8. Génération de la soumission finale

In [ ]:
trustii_ids = df_test['trustii_id'].values

submission = pd.DataFrame({
    'trustii_id'        : trustii_ids,
    'risk_hepatic_event': test_hep,
    'risk_death'        : test_dth,
})

submission.to_csv(OUTPUT_CSV, index=False)

print(f'Soumission sauvegardée : {OUTPUT_CSV}')
print(f'Patients : {len(submission)}')
print(f'\nAperçu :')
print(submission.head(10).to_string(index=False))

print(f'\nStatistiques des prédictions :')
print(submission[['risk_hepatic_event','risk_death']].describe().round(4).to_string())

## 9. Conclusion

### Pipeline final

**XGBoost Cox multi-seed** — feature engineering temporel enrichi :
- 162 features extraites depuis les trajectoires longitudinales (vs 35-44 features flat dans la baseline)
- L'objectif `survival:cox` natif gère correctement les données censurées
- 5-fold CV stratifié + 5 seeds → prédictions stables (bagging)
- **Score OOF estimé : 0.882**

### Points clés

1. **FibroScan est le marqueur dominant** — la rigidité hépatique maximale et son accélération
   prédit mieux que les valeurs ponctuelles. Cela confirme l'importance du **suivi longitudinal**.

2. **FIB-4 index** (age×AST)/(plt×√ALT) — marqueur clinique standard retrouvé en top feature,
   ce qui valide la cohérence clinique du modèle.

3. **L'accélération temporelle** (variation entre 1ère et 2e moitié du suivi) capture
   la dynamique de progression que les statistiques classiques (mean, std) ne voient pas.

4. **Le SSM K-Mamba** (modèle séquentiel C pur) atteint 0.808 OOF en 50 epochs —
   complémentaire pour capturer les trajectoires complexes, avec potentiel de progression
   supplémentaire avec plus d'epochs d'entraînement.